# Task 5: RAG System with LangChain (Kaggle, no API key)
**Goal:** answer questions about a PDF or Excel sheet using Retrieval-Augmented Generation.

Pipeline: **Load** → **Split** → **Embed** → **Vector store** → **Retriever** → **Prompt + LLM (generator)**

## Before you run (Kaggle settings)
This notebook uses a **free open-source model that runs inside Kaggle**, so **no API key is needed**.
In the right sidebar → **Session options**:
1. **Accelerator** → *GPU T4 x2* (or *GPU P100*), because on CPU it will be very slow.
2. **Internet** → **On** (needed once to download the model; Kaggle may ask you to verify your phone number).

## 1. Install

In [41]:
!pip install -q langchain-core langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu pypdf openpyxl accelerate

## 2. Load the free LLM

In [42]:
# Free local LLM - NO API key. Runs on the Kaggle GPU.
import torch, transformers
from transformers import pipeline, AutoTokenizer
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableLambda

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"     # smaller/faster option: "Qwen/Qwen2.5-1.5B-Instruct"

transformers.logging.set_verbosity_error()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
pipe = pipeline("text-generation", model=MODEL_ID, tokenizer=tokenizer,
                torch_dtype=torch.float16, device_map="auto")
print("Model loaded on:", pipe.model.device)

_ROLES = {"system": "system", "human": "user", "ai": "assistant"}

def get_llm(temperature=0.3, max_new_tokens=700):
    """Returns a LangChain runnable: accepts a prompt / list of messages, returns an AIMessage."""
    def _run(x):
        if isinstance(x, str):
            msgs = [HumanMessage(content=x)]
        elif hasattr(x, "to_messages"):
            msgs = x.to_messages()
        else:
            msgs = x
        chat = [{"role": _ROLES.get(m.type, "user"), "content": m.content} for m in msgs]
        prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        kwargs = dict(max_new_tokens=max_new_tokens, return_full_text=False)
        if temperature > 0:
            kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
        else:
            kwargs.update(do_sample=False)
        out = pipe(prompt, **kwargs)[0]["generated_text"]
        return AIMessage(content=out.strip())
    return RunnableLambda(_run)

def text_of(content):
    return content if isinstance(content, str) else "".join(b.get("text", "") for b in content if isinstance(b, dict))

print(get_llm(0).invoke("Say hello in one short sentence.").content)   # quick test

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded on: cuda:0
Hello!


## 3. Load the knowledge base
Add your file with **+ Add Input → Upload** (new dataset) in the Kaggle sidebar. It will appear under `/kaggle/input/`.
Leave `FILE_PATH = None` to auto-detect the first PDF/Excel/CSV found there, or set the path yourself.

In [43]:
from pathlib import Path
import glob
import pandas as pd
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

FILE_PATH = None    # e.g. "/kaggle/input/my-dataset/lecture.pdf"

if FILE_PATH is None:
    found = [f for ext in ("pdf", "xlsx", "xls", "csv")
             for f in glob.glob(f"/kaggle/input/**/*.{ext}", recursive=True)]
    print("Files found:", found)
    if not found:
        raise SystemExit("No file found. Add your PDF/Excel file via 'Add Input' in the sidebar.")
    FILE_PATH = found[0]
print("Using:", FILE_PATH)

def load_documents(path):
    p = Path(path)
    if p.suffix.lower() == ".pdf":
        return PyPDFLoader(str(p)).load()          # one Document per page
    if p.suffix.lower() in {".xlsx", ".xls", ".csv"}:
        sheets = {"csv": pd.read_csv(p)} if p.suffix.lower() == ".csv" else pd.read_excel(p, sheet_name=None)
        docs = []
        for sheet, df in sheets.items():
            df = df.dropna(how="all").fillna("")
            for i, row in df.iterrows():           # each row -> one readable record
                text = "; ".join(f"{c}: {v}" for c, v in row.items() if str(v) != "")
                docs.append(Document(page_content=text, metadata={"source": p.name, "sheet": sheet, "row": int(i) + 2}))
        return docs
    raise ValueError("Use a .pdf, .xlsx, .xls or .csv file")

docs = load_documents(FILE_PATH)
print(f"Loaded {len(docs)} document(s)")
print(docs[0].page_content[:300])

Files found: ['/kaggle/input/datasets/roqayamuhammad/propmate-ai-project-specification/PropMate_AI_Project_Specification.pdf']
Using: /kaggle/input/datasets/roqayamuhammad/propmate-ai-project-specification/PropMate_AI_Project_Specification.pdf
Loaded 2 document(s)
PropMate AI: Intelligent Operational Assistant for
Real Estate Customer Care
Graduation Project Specification & Architectural Blueprint | Full English Edition
Project Name: PropMate AI
Domain: Real Estate PropTech / Multi-Agent RAG
Core Focus: Proactive SLA Breach Prevention
Status: Validated Specif


## 4. Split, embed and index (the retriever)

In [44]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)
print(f"{len(chunks)} chunks")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

6 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 5. Build the generator and the RAG chain

In [45]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I couldn't find that in the document."
Keep the answer clear and concise.

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | get_llm(temperature=0)
    | StrOutputParser()
)

def source_label(d):
    m = d.metadata
    if "page" in m:
        return f"{Path(m.get('source','')).name} (page {m['page'] + 1})"
    return f"{m.get('source')} (sheet {m.get('sheet')}, row {m.get('row')})"

def ask(question):
    print("Q:", question)
    print("A:", rag_chain.invoke(question))
    print("Sources:", ", ".join(sorted({source_label(d) for d in retriever.invoke(question)})), "\n")

## 6. Ask questions
Try questions from inside your file, plus one that is NOT in the file. It should answer *I couldn't find that in the document*.

In [46]:
ask("What is this document about?")
ask("Summarize the key points.")
ask("Who won the 2018 World Cup?")   # not in the document -> should say it couldn't find it

Q: What is this document about?
A: This document describes a multi-agent system architecture for a PropMate Real Estate Compound, focusing on preventing SLA violations proactively using predictive machine learning combined with language models. It covers database setup, ML model training, RAG indexing, multi-agent integration, and dashboard development.
Sources: PropMate_AI_Project_Specification.pdf (page 1), PropMate_AI_Project_Specification.pdf (page 2) 

Q: Summarize the key points.
A: Key Points:

1. **Database Setup & ML Model**: Ingested 231k CSV rows into SQLite, performed feature engineering, trained a LightGBM classifier, and exported the model.
2. **RAG Indexing & Triage Setup**: Indexed 3 Markdown SOPs into ChromaDB, engineered prompts for Arabic complaint parsing, and grounded response generation.
3. **Multi-Agent Graph Integration**: Built a stateful agent graph in LangGraph linking triage, risk evaluation, and RAG agents, and implemented a Text-to-SQL pipeline.
4. **Dashb

In [ ]:
# Interactive mode: type 'exit' to stop
while (q := input("Your question: ").strip()).lower() not in {"exit", "quit", ""}:
    ask(q)